# Day 17 - A managed training job with built-in XGBoost

Run on the SageMaker notebook instance, kernel `conda_python3`. **Nothing here needs pip installs** -
it's boto3 only.

The big idea: this notebook doesn't train anything. It sends one API call - `CreateTrainingJob` - and
SageMaker spins up a *separate* `ml.m5.large`, pulls AWS's XGBoost container, copies the data from S3,
trains, writes `model.tar.gz` back to S3, and shuts the machine down. You pay only for those minutes.
The notebook could be stopped the moment the job starts and the job would still finish.

We call the raw API with boto3 rather than the SageMaker Python SDK's `Estimator`. The SDK is a
convenience wrapper around this exact call - seeing the call itself is what makes the wrapper make sense.


In [ ]:
import json, time
import boto3

REGION = "ca-central-1"
BUCKET = "dave-ds-lab-ca"

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

# The role this notebook runs as - the training job will run as the same role.
ROLE = sm.describe_notebook_instance(NotebookInstanceName="ds-lab-notebook")["RoleArn"]

# AWS-owned image for the built-in XGBoost algorithm in ca-central-1.
# Account 341280168497 hosts SageMaker's first-party algorithm images in this region.
IMAGE = f"341280168497.dkr.ecr.{REGION}.amazonaws.com/sagemaker-xgboost:1.7-1"

print("Role :", ROLE)
print("Image:", IMAGE)

## 1. Check the channels exist

Created by `sql/04_xgb_channels.sql` in Athena. CSV, target first, no header.


In [ ]:
for ch in ["train", "validation"]:
    objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"features/xgb/{ch}/").get("Contents", [])
    total = sum(o["Size"] for o in objs)
    print(f"{ch:10s} {len(objs)} file(s)  {total/1e6:7.1f} MB")

first = s3.list_objects_v2(Bucket=BUCKET, Prefix="features/xgb/train/")["Contents"][0]["Key"]
head = s3.get_object(Bucket=BUCKET, Key=first, Range="bytes=0-300")["Body"].read().decode()
print("\nfirst lines of train:\n" + "\n".join(head.splitlines()[:3]))

## 2. Describe the job

Everything a training job needs, in one dictionary. Read it top to bottom - each block is a question
SageMaker must have answered: *which code? which data? which machine? where do results go? when to give up?*

Hyperparameters are always strings in this API. `early_stopping_rounds` watches `validation:mae`;
that makes March a little less "unseen" than on Day 16 - it picked the stopping round. Worth saying out loud
in an interview; the fix is a third split (e.g. train Jan, early-stop Feb, report Mar).


In [ ]:
JOB = "xgb-tips-" + time.strftime("%Y%m%d-%H%M%S")

job = dict(
    TrainingJobName=JOB,
    RoleArn=ROLE,
    AlgorithmSpecification={"TrainingImage": IMAGE, "TrainingInputMode": "File"},
    HyperParameters={
        "objective": "reg:squarederror",
        "eval_metric": "mae",
        "num_round": "500",
        "early_stopping_rounds": "20",
        "max_depth": "8",
        "eta": "0.1",
        "subsample": "0.8",
        "colsample_bytree": "0.8",
        "min_child_weight": "50",
        "tree_method": "hist",
    },
    InputDataConfig=[
        {
            "ChannelName": ch,
            "ContentType": "text/csv",
            "DataSource": {"S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": f"s3://{BUCKET}/features/xgb/{ch}/",
                "S3DataDistributionType": "FullyReplicated",
            }},
        }
        for ch in ["train", "validation"]
    ],
    OutputDataConfig={"S3OutputPath": f"s3://{BUCKET}/models/builtin/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 10},
    # Safety cap: if anything hangs, SageMaker kills the job after 30 minutes. Always set this.
    StoppingCondition={"MaxRuntimeInSeconds": 1800},
    Tags=[{"Key": "project", "Value": "ds-lab"}],
)
print(json.dumps(job["HyperParameters"], indent=2))

## 3. Launch it


In [ ]:
sm.create_training_job(**job)
print("launched", JOB)
print(f"console: https://{REGION}.console.aws.amazon.com/sagemaker/home?region={REGION}#/jobs/{JOB}")

## 4. Watch the lifecycle

The job moves through the same stages every time: **Starting** (finding a machine) -> **Downloading**
(image + data) -> **Training** -> **Uploading** (model.tar.gz to S3) -> **Completed**. Most of a short
job's wall-clock is the first two - which is why tiny experiments are faster in a notebook, and big ones
are cheaper as jobs.


In [ ]:
last = None
t0 = time.time()
while True:
    d = sm.describe_training_job(TrainingJobName=JOB)
    status, secondary = d["TrainingJobStatus"], d["SecondaryStatus"]
    if secondary != last:
        print(f"{time.time()-t0:6.0f}s  {status:11s} {secondary}")
        last = secondary
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)

if status != "Completed":
    print("\nFailureReason:", d.get("FailureReason"))

## 5. What it cost and what it produced


In [ ]:
d = sm.describe_training_job(TrainingJobName=JOB)
billable = d.get("BillableTimeInSeconds", 0)
PRICE_PER_HOUR = 0.134   # ml.m5.large training, ca-central-1, approx - check the pricing page

print("Artifact        :", d["ModelArtifacts"]["S3ModelArtifacts"])
print("Training time   :", d.get("TrainingTimeInSeconds"), "s")
print("Billable time   :", billable, "s")
print(f"Approx cost     : ${billable / 3600 * PRICE_PER_HOUR:.4f}")
print()
for m in d.get("FinalMetricDataList", []):
    print(f"{m['MetricName']:20s} {m['Value']:.4f}")

## 6. Last lines of the job's own log

Every training job writes to CloudWatch Logs, group `/aws/sagemaker/TrainingJobs`. When a job fails,
this is where the real error is - `FailureReason` is usually just the last line.


In [ ]:
logs = boto3.client("logs", region_name=REGION)
GROUP = "/aws/sagemaker/TrainingJobs"
streams = logs.describe_log_streams(logGroupName=GROUP, logStreamNamePrefix=JOB)["logStreams"]
for s in streams:
    events = logs.get_log_events(logGroupName=GROUP, logStreamName=s["logStreamName"], startFromHead=False, limit=15)
    for e in events["events"]:
        print(e["message"])

## 7. Compare with Day 16

Day 16: sklearn on a 1.5M-row sample, trained inside the notebook. Today: XGBoost on all 4.6M training rows,
trained on a machine that existed only for this job.


In [ ]:
manual = json.loads(s3.get_object(Bucket=BUCKET, Key="models/manual/metrics.json")["Body"].read())
builtin_mae = next(m["Value"] for m in d["FinalMetricDataList"] if m["MetricName"] == "validation:mae")

print(f"Day 16  sklearn HGB, notebook, 1.5M rows : MAE ${manual['valid']['hgb']['mae']:.3f}")
print(f"Day 17  built-in XGBoost job, 4.6M rows   : MAE ${builtin_mae:.3f}")
print(f"One-line rule (zone rate x fare)          : MAE ${manual['valid']['baseline_zone_rate_x_fare']['mae']:.3f}")

## 8. Nothing to stop for the job - it's already gone

The training instance shut itself down at *Completed*. Check: SageMaker console -> Training -> Training jobs:
status Completed, and nothing under Inference -> Endpoints.

**Do stop this notebook instance** when you're done: Notebooks -> `ds-lab-notebook` -> Actions -> Stop.
